# 🤗 Hugging Face Basics - Getting Started with AI Models

This notebook introduces you to the fundamentals of using Hugging Face Transformers for various AI tasks.

## What you'll learn:
- How to load and use pre-trained models
- Text classification (sentiment analysis)
- Text generation
- Question answering
- Working with different model types

## Prerequisites:
Make sure you've installed the required packages:
```bash
pip install transformers torch datasets
```

## 1. Setup and Imports

In [ ]:
# Import required libraries
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForQuestionAnswering, AutoModelForCausalLM,
    pipeline, set_seed
)
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
set_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Sentiment Analysis with Pipelines

The easiest way to get started with Hugging Face is using pipelines. They handle tokenization, model inference, and post-processing automatically.

In [ ]:
# Create a sentiment analysis pipeline
print("Loading sentiment analysis model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    return_all_scores=True
)
print("✅ Model loaded!")

In [ ]:
# Test with some example texts
texts = [
    "I absolutely love this product! It's amazing!",
    "This is the worst thing I've ever bought.",
    "It's okay, nothing special.",
    "I'm feeling great about this new opportunity!",
    "The weather is nice today."
]

print("🔍 Analyzing sentiment for sample texts:\n")
for i, text in enumerate(texts, 1):
    result = sentiment_pipeline(text)
    
    # Get the highest scoring sentiment
    best_result = max(result[0], key=lambda x: x['score'])
    
    print(f"{i}. Text: \"{text}\"")
    print(f"   Sentiment: {best_result['label']} (Score: {best_result['score']:.3f})")
    
    # Show all scores
    all_scores = ", ".join([f"{item['label']}: {item['score']:.3f}" for item in result[0]])
    print(f"   All scores: {all_scores}\n")

## 3. Manual Model Loading (Lower-level API)

For more control, you can load models and tokenizers separately.

In [ ]:
# Load model and tokenizer manually
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f"✅ Model loaded!")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Tokenizer vocab size: {len(tokenizer.vocab):,}")

In [ ]:
# Manual prediction function
def predict_sentiment_manual(text, model, tokenizer):
    # Tokenize the input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Convert to probabilities
    probabilities = predictions[0].tolist()
    
    # Map to labels (for this specific model)
    labels = ['NEGATIVE', 'POSITIVE']
    
    return {label: prob for label, prob in zip(labels, probabilities)}

# Test the manual prediction
test_text = "I love machine learning and AI!"
result = predict_sentiment_manual(test_text, model, tokenizer)

print(f"Text: \"{test_text}\"")
print(f"Predictions: {result}")
print(f"Dominant sentiment: {max(result, key=result.get)} ({max(result.values()):.3f})")

## 4. Text Generation

Let's try generating text with a language model.

In [ ]:
# Create a text generation pipeline
print("Loading text generation model (this might take a moment)...")
generator = pipeline(
    "text-generation",
    model="gpt2",  # Using GPT-2 small model
    tokenizer="gpt2"
)
print("✅ Text generation model loaded!")

In [ ]:
# Generate text with different prompts
prompts = [
    "Artificial intelligence will",
    "The future of technology is",
    "In the world of machine learning,"
]

print("🤖 Generating text completions:\n")
for i, prompt in enumerate(prompts, 1):
    print(f"{i}. Prompt: \"{prompt}\"")
    
    # Generate text
    results = generator(
        prompt,
        max_length=50,
        num_return_sequences=2,
        temperature=0.7,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    
    for j, result in enumerate(results, 1):
        generated_text = result['generated_text'][len(prompt):].strip()
        print(f"   Option {j}: {generated_text}")
    print()

## 5. Question Answering

Let's try extractive question answering - finding answers within a given context.

In [ ]:
# Create a question answering pipeline
print("Loading question answering model...")
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)
print("✅ Question answering model loaded!")

In [ ]:
# Define context and questions
context = """
Hugging Face is a company that develops tools for building applications using machine learning. 
It is most notable for its transformers library built for natural language processing applications 
and its platform that allows users to share machine learning models and datasets. The company was 
founded in 2016 by Clément Delangue, Julien Chaumond, and Thomas Wolf. Hugging Face is based in 
New York City and Paris, with additional offices in various locations worldwide.
"""

questions = [
    "What is Hugging Face?",
    "When was Hugging Face founded?",
    "Who founded Hugging Face?",
    "Where is Hugging Face based?",
    "What is the transformers library used for?"
]

print("❓ Question Answering Results:\n")
print(f"Context: {context.strip()}\n")
print("-" * 60)

for i, question in enumerate(questions, 1):
    result = qa_pipeline({
        'question': question,
        'context': context
    })
    
    print(f"{i}. Q: {question}")
    print(f"   A: {result['answer']} (Score: {result['score']:.3f})")
    print()

## 6. Working with Different Model Types

Let's explore what different models are available and their use cases.

In [ ]:
# Dictionary of popular models and their use cases
popular_models = {
    "Text Classification": [
        "cardiffnlp/twitter-roberta-base-sentiment-latest",
        "distilbert-base-uncased-finetuned-sst-2-english",
        "microsoft/DialoGPT-medium"
    ],
    "Text Generation": [
        "gpt2",
        "microsoft/DialoGPT-medium",
        "EleutherAI/gpt-neo-1.3B"
    ],
    "Question Answering": [
        "distilbert-base-cased-distilled-squad",
        "deepset/roberta-base-squad2",
        "microsoft/DialoGPT-medium"
    ],
    "Named Entity Recognition": [
        "dbmdz/bert-large-cased-finetuned-conll03-english",
        "dslim/bert-base-NER"
    ],
    "Translation": [
        "Helsinki-NLP/opus-mt-en-fr",
        "Helsinki-NLP/opus-mt-fr-en",
        "facebook/m2m100_418M"
    ],
    "Summarization": [
        "facebook/bart-large-cnn",
        "t5-small",
        "google/pegasus-xsum"
    ]
}

print("🤗 Popular Hugging Face Models by Task:\n")
for task, models in popular_models.items():
    print(f"📋 {task}:")
    for model in models:
        print(f"   • {model}")
    print()

## 7. Performance Tips and Best Practices

In [ ]:
import time

# Performance comparison: Pipeline vs Manual
test_texts = ["This is a great day!"] * 100

print("⚡ Performance Comparison:\n")

# Time pipeline approach
start_time = time.time()
for text in test_texts:
    _ = sentiment_pipeline(text)
pipeline_time = time.time() - start_time

print(f"Pipeline approach: {pipeline_time:.2f} seconds for {len(test_texts)} texts")
print(f"Average per text: {pipeline_time/len(test_texts)*1000:.1f} ms")

# Time manual approach with batching
start_time = time.time()
inputs = tokenizer(test_texts, return_tensors="pt", truncation=True, padding=True)
with torch.no_grad():
    outputs = model(**inputs)
manual_time = time.time() - start_time

print(f"\nManual batch approach: {manual_time:.2f} seconds for {len(test_texts)} texts")
print(f"Average per text: {manual_time/len(test_texts)*1000:.1f} ms")
print(f"\n🚀 Batching is {pipeline_time/manual_time:.1f}x faster!")

## 8. Model Information and Exploration

In [ ]:
# Function to analyze model information
def analyze_model(model_name):
    """Analyze and display information about a model."""
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"📊 Model Analysis: {model_name}")
        print(f"   Total parameters: {total_params:,}")
        print(f"   Trainable parameters: {trainable_params:,}")
        print(f"   Vocabulary size: {len(tokenizer.vocab):,}")
        print(f"   Max sequence length: {tokenizer.model_max_length}")
        print(f"   Model type: {model.config.model_type}")
        if hasattr(model.config, 'num_labels'):
            print(f"   Number of labels: {model.config.num_labels}")
        print()
        
    except Exception as e:
        print(f"❌ Error analyzing {model_name}: {str(e)}")

# Analyze some models
models_to_analyze = [
    "distilbert-base-uncased-finetuned-sst-2-english",
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
]

for model_name in models_to_analyze:
    analyze_model(model_name)

## 9. Saving and Loading Custom Models

In [ ]:
# Save a model locally
save_directory = "./my_saved_model"

print(f"💾 Saving model to {save_directory}...")
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print("✅ Model saved!")

# Load the saved model
print(f"📂 Loading model from {save_directory}...")
loaded_tokenizer = AutoTokenizer.from_pretrained(save_directory)
loaded_model = AutoModelForSequenceClassification.from_pretrained(save_directory)
print("✅ Model loaded!")

# Test that it works
test_text = "This saved model works perfectly!"
result = predict_sentiment_manual(test_text, loaded_model, loaded_tokenizer)
print(f"\n🧪 Test with loaded model:")
print(f"Text: \"{test_text}\"")
print(f"Result: {result}")

## 10. Next Steps and Resources

Congratulations! You've learned the basics of working with Hugging Face Transformers. Here are some next steps:

### 🚀 Next Steps:
1. **Fine-tune a model** on your own dataset
2. **Deploy models** using the web app templates
3. **Explore advanced features** like model compression and quantization
4. **Try multimodal models** for images and text

### 📚 Helpful Resources:
- [Hugging Face Course](https://huggingface.co/course)
- [Transformers Documentation](https://huggingface.co/docs/transformers)
- [Model Hub](https://huggingface.co/models)
- [Datasets Hub](https://huggingface.co/datasets)

### 💡 Project Ideas:
- Build a news sentiment analyzer
- Create a chatbot with personality
- Make a text summarization tool
- Build a multilingual translator
- Create an automated content moderator

In [ ]:
# Cleanup
import shutil

# Remove the saved model directory
if os.path.exists(save_directory):
    shutil.rmtree(save_directory)
    print(f"🧹 Cleaned up saved model directory: {save_directory}")

print("\n🎉 Tutorial complete! You're ready to build amazing AI applications!")